In [1]:
import pandas as pd

# Загрузка файла
df = pd.read_csv('ДЗ_Pandas.csv')

In [2]:
# 1.1 Определить, в каких столбцах есть пропущенные значения
print(df.isnull().sum())



ID                 0
Название фильма    0
Год выпуска        0
Жанр               0
Рейтинг            2
Бюджет (млн $)     2
Сборы (млн $)      0
Режиссер           2
Основные актеры    0
dtype: int64


In [3]:
# 1.2 Удалить строки, где пропущено значение в 'Бюджет (млн $)'
df = df.dropna(subset=['Бюджет (млн $)'])
print(df['Бюджет (млн $)'].unique())


[120.  90.  60.  85.  40. 130.  95.  65.]


In [4]:
# 1.3 Заполнить пропуски в 'Режиссер' строкой 'Неизвестен'
df['Режиссер'] = df['Режиссер'].fillna('Неизвестен')
print(df['Режиссер'].fillna)

<bound method NDFrame.fillna of 0        Неизвестен
1    Мария Смирнова
2     Алексей Попов
3    Кирилл Соколов
5     Ольга Петрова
6        Неизвестен
7    Роман Васильев
9    Дмитрий Иванов
Name: Режиссер, dtype: object>


In [5]:
# 1.4 Заполнить пропуски в 'Рейтинг' средним значением
df['Рейтинг'] = df['Рейтинг'].fillna(df['Рейтинг'].mean())
print(df['Рейтинг'].fillna)

<bound method NDFrame.fillna of 0    7.80
1    8.20
2    8.50
3    7.60
5    7.85
6    7.85
7    7.90
9    7.10
Name: Рейтинг, dtype: float64>


In [6]:
# 1.5 Удалить строки, где пропущены и 'Бюджет (млн $)', и 'Сборы (млн $)'
# (т.к. 'Бюджет' уже очищен выше, эта проверка формальная, если бы было раньше)
df = df[~(df['Бюджет (млн $)'].isnull() & df['Сборы (млн $)'].isnull())]
print(df['Бюджет (млн $)'].isnull)

<bound method Series.isnull of 0    120.0
1     90.0
2     60.0
3     85.0
5     40.0
6    130.0
7     95.0
9     65.0
Name: Бюджет (млн $), dtype: float64>


In [7]:
# Задание 2 - Grouby
# Группировка по Жанру:
 # Сгруппируйте фильмы по жанру
 # Рассчитайте средний рейтинг и средний бюджет в каждом жанре.
genre_group = df.groupby('Жанр')[['Рейтинг', 'Бюджет (млн $)']].mean()
print(genre_group)


                    Рейтинг  Бюджет (млн $)
Жанр                                       
Боевик                 7.75            90.0
Драма                  8.50            60.0
Мистика                7.10            65.0
Научная фантастика     7.85           130.0
Приключения            8.20            90.0
Триллер                7.85            40.0
Фантастика             7.80           120.0


In [8]:
# Группировка по Году Выпуска:
# Сгруппируйте фильмы по году выпуска
# Определите общее количество фильмов и суммарные сборы в прокате по каждому году.

year_group = df.groupby('Год выпуска').agg({
    'Название фильма': 'count',  # количество фильмов
    'Сборы (млн $)': 'sum'  # суммарные сборы
}).rename(columns={'Название': 'Количество фильмов'})
print(year_group)


             Название фильма  Сборы (млн $)
Год выпуска                                
2018                       1            200
2019                       1            550
2020                       1            450
2021                       2            780
2022                       2            810
2023                       1            650


In [9]:
# Комбинированная Группировка:
#  Сгруппируйте фильмы по жанру и году выпуска
# Рассчитайте средний рейтинг для каждой комбинации жанра и года выпуска
genre_year_group = df.groupby(['Жанр', 'Год выпуска'])['Рейтинг'].mean()
print(genre_year_group)


Жанр                Год выпуска
Боевик              2019           7.60
                    2021           7.90
Драма               2021           8.50
Мистика             2022           7.10
Научная фантастика  2023           7.85
Приключения         2020           8.20
Триллер             2018           7.85
Фантастика          2022           7.80
Name: Рейтинг, dtype: float64


In [10]:
# Задание 3 - reset index
# Группировка и Сортировка:
# Сгруппируйте фильмы по жанру и рассчитайте средний рейтинг для каждого жанра
# Сбросьте индекс и отсортируйте результаты по среднему рейтингу в убывающем порядке
avg_rating = df.groupby('Жанр')['Рейтинг'].mean().reset_index().sort_values(by='Рейтинг', ascending=False)
print(avg_rating)


                 Жанр  Рейтинг
1               Драма     8.50
4         Приключения     8.20
3  Научная фантастика     7.85
5             Триллер     7.85
6          Фантастика     7.80
0              Боевик     7.75
2             Мистика     7.10


In [11]:
# Анализ частоты:
# Используйте value_counts для определения, сколько фильмов было выпущено в каждом году
# Отсортируйте результаты по годам в порядке возрастания

release_count = df['Год выпуска'].value_counts().sort_index()
print(release_count)


Год выпуска
2018    1
2019    1
2020    1
2021    2
2022    2
2023    1
Name: count, dtype: int64


In [12]:
# Комбинированный Анализ:
# Сгруппируйте фильмы по режиссеру и жанру, затем рассчитайте общие сборы фильмов для каждой комбинации
# Сбросьте индекс и отсортируйте результаты по сумме сборов в убывающем порядке

revenue_group = df.groupby(['Режиссер', 'Жанр'])['Сборы (млн $)'].sum().reset_index()
revenue_sorted = revenue_group.sort_values(by='Сборы (млн $)', ascending=False)
print(revenue_sorted)


         Режиссер                Жанр  Сборы (млн $)
4      Неизвестен  Научная фантастика            650
2  Кирилл Соколов              Боевик            550
5      Неизвестен          Фантастика            500
7  Роман Васильев              Боевик            480
3  Мария Смирнова         Приключения            450
1  Дмитрий Иванов             Мистика            310
0   Алексей Попов               Драма            300
6   Ольга Петрова             Триллер            200
